In [15]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from pybamm import exp
from pybamm import tanh

Defining Model Variables

In [16]:

xi = pybamm.SpatialVariable(
    "xi", domain="SEI layer", coord_sys="cartesian")

CL_atom = pybamm.Variable(
    "concentration of netutral lithium atoms in the SEI [mol.m-3]",  domain="SEI layer")
CL_ion = pybamm.Variable(
    "concentration of llithium ions in the SEI [mol.m-3]",  domain="SEI layer")
Phi_SEI = pybamm.Variable("Potential in the SEI [V]",  domain="SEI layer")
L_SEI = pybamm.Variable("Thickness of SEI [m]")

In [17]:
model = pybamm.lithium_ion.BaseModel()

Defining parameters of the model

In [18]:
T = pybamm.Parameter('Initial temperature [K]')
R = pybamm.Parameter('Ideal gas constant [J.K-1.mol-1]')
F = pybamm.Parameter("Faraday constant [C.mol-1]")
D_Li_atom = pybamm.Parameter(
    'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]')
D_Li_ion = pybamm.Parameter(
    'diffusion coefficient of llithium ions in the SEI [m2.s-1]')

CL_atom_0 = pybamm.Parameter(
    "initial concentration of netutral lithium atoms in the SEI [mol.m-3]")
CL_ion_0 = pybamm.Parameter(
    "initial concentration of llithium ions in the SEI [mol.m-3]")

Phi_SEI_0 = pybamm.Parameter("initial potential in the SEI [V]")


M_SEI = pybamm.Parameter("molar weight of SEI material [kg.mol-1]")
rho_SEI = pybamm.Parameter("density of SEI material [kg.m-3]")
n_SEI = pybamm.Parameter("electrone number [-]")


L_tun = pybamm.Parameter("length of the tunneling [m]")
L_SEI_0 = pybamm.Parameter("initial thickness of SEI [m]")

CL_ion_max = pybamm.Parameter(
    "maximum concentration of llithium ions in the electrolyte [mol.m-3]")

j0_0 = pybamm.Parameter(
    "Butler-Volmer rate constant for intercalation [A.m-2]")

J_total = pybamm.Parameter("Total current density [A.m-2]")

j_sei_0 = pybamm.Parameter("rate constant SEI [A.m-2]")


def Un(x):
    return pybamm.FunctionParameter(
        "Negative electrode OCP [V]",
        {"Negative particle stoichiometry": x},
    )

In [19]:
# def OCP_n(sto):
#     stretch = 1.00
#     # sto = stretch * cc / c_n_max
#     u_eq = (
#         1.9793 * exp(-39.3631 * sto)
#         + 0.2482
#         - 0.0909 * tanh(29.8538 * (sto - 0.1234))
#         - 0.04478 * tanh(14.9159 * (sto - 0.2769))
#         - 0.0205 * tanh(30.4444 * (sto - 0.6103))
#     )
#     return u_eq


def OCP_n(sto):
    return ((1.24-sto)/1.16)**2.92

In [20]:
param = pybamm.ParameterValues(
    {
        'Initial temperature [K]': 298.15,
        'Ideal gas constant [J.K-1.mol-1]': 8.314462618,
        "Faraday constant [C.mol-1]": 96485.33212,
        # Kolzenberg
        'diffusion coefficient of netutral lithium atoms in the SEI [m2.s-1]': 1e-15,
        # pybamm (also not sure)
        'diffusion coefficient of llithium ions in the SEI [m2.s-1]': 3.3e-14,
        # Kolzenberg
        'initial concentration of netutral lithium atoms in the SEI [mol.m-3]': 1000,
        # pybamm
        'initial concentration of llithium ions in the SEI [mol.m-3]': 33000,
        'initial potential in the SEI [V]': 0,
        'molar weight of SEI material [kg.mol-1]': 0.162,
        'density of SEI material [kg.m-3]': 1690,
        'electrone number [-]': 2,
        'length of the tunneling [m]': 2e-9,
        'initial thickness of SEI [m]': 1e-12,
        'maximum concentration of llithium ions in the electrolyte [mol.m-3]': 33000,
        # Kolzenberg
        'Butler-Volmer rate constant for intercalation [A.m-2]': 6.4e-7,
        "Total current density [A.m-2]": -2,
        "rate constant SEI [A.m-2]": 7.04e-5,  # Kolzenberg
        "Negative electrode OCP [V]": OCP_n,


    }
)

In [21]:
NL_ions = - D_Li_ion / L_SEI * pybamm.grad(CL_ion)
# - D_Li_ion * F/(  R*T * L_SEI)*pybamm.grad(Phi_SEI)  # define the flux for lithoum ions
# define the flux for netutral lithium atoms
NL_atom = - D_Li_atom / L_SEI * pybamm.grad(CL_atom)


V_SEI = M_SEI/(n_SEI*rho_SEI)

# Unclear how to define the source terms
A = 1
R_CLi_ions = 0
R_CLi_atom = 0


# J_Li_0 = NL_atom - D_Li_atom*F/(R*T * L_SEI)*CL_atom*pybamm.grad(Phi_SEI)
# J_tun = A * J_Li_0 * pybamm.exp(- L_SEI / L_tun)
# Je = J_Li_0 + J_tun
eta_sei = Phi_SEI - 0.1
alpha = 0.5
J_sei = - j_sei_0 * pybamm.exp(-alpha*F/(R*T)*eta_sei)

# define the rhs equation
dL_SEI_dt = -V_SEI / F * pybamm.BoundaryValue(J_sei, "left")
dCL_ion_dt = -1 / L_SEI * pybamm.div(NL_ions) + \
    R_CLi_ions  # define the rhs equation
dCL_atom_dt = -1 / L_SEI * pybamm.div(NL_atom) + \
    R_CLi_atom  # define the rhs equation

In [22]:

eta_int = Phi_SEI - Un(CL_ion/CL_ion_max)
# Prescribed current density
J_int = 2*j0_0 * pybamm.sinh(F/(2*R*T) * eta_int)
model.algebraic = {Phi_SEI: J_total - (J_int+J_sei)}
model.rhs = {CL_ion: dCL_ion_dt, CL_atom: dCL_atom_dt, L_SEI: dL_SEI_dt}

In [23]:
model.variables = {
    "concentration of netutral lithium atoms in the SEI [mol.m-3]": CL_atom,
    "concentration of llithium ions in the SEI [mol.m-3]": CL_ion,
    "Potential in the SEI [V]": Phi_SEI,
    "Thickness of SEI [m]": L_SEI
}

In [24]:
model.initial_conditions = {CL_ion: CL_ion_0, CL_atom: CL_atom_0,
                            Phi_SEI: Phi_SEI_0, L_SEI: L_SEI_0}

lbc_CL_ion = pybamm.BoundaryValue(J_int, "left")/(F*D_Li_ion)
rbc_CL_ion = CL_ion_max

lbc_CL_atom = pybamm.BoundaryValue(J_sei, "left")/(F*D_Li_atom)
rbc_CL_atom = 0

model.boundary_conditions = {CL_ion: {"left": (lbc_CL_ion, "Neumann"), "right": (rbc_CL_ion, "Dirichlet")},
                             CL_atom: {"left": (lbc_CL_atom, "Neumann"), "right": (rbc_CL_atom, "Dirichlet")}}

In [25]:

geometry = pybamm.Geometry(
    {"SEI layer": {xi: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(1)}}})

In [26]:
param.process_model(model)
param.process_geometry(geometry)
submesh_types = {"SEI layer": pybamm.Uniform1DSubMesh}
var_pts = {xi: 50}
# # create a mesh of our geometry, using a uniform grid with 20 volumes
mesh = pybamm.Mesh(geometry, submesh_types, var_pts)
spatial_methods = {"SEI layer": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)
disc.process_model(model)

In [27]:
# solver = pybamm.ScipySolver()
# pybamm.CasadiSolver(mode="fast")
pybamm.settings.max_y_value = 1000000000
solver = pybamm.IDAKLUSolver()

In [28]:
sim = pybamm.Simulation(
    model,
    geometry=geometry,
    parameter_values=param,
    var_pts=var_pts,
    spatial_methods=spatial_methods,
    solver=solver,
)

sol = sim.solve([0, 100])

CasADi - 2024-04-05 22:14:13 WARNING("roots:g failed: Inf detected for output x, at (row 0, col 0).") [.../casadi/core/oracle_function.cpp:377]
CasADi - 2024-04-05 22:14:13 WARNING("roots:g failed: Inf detected for output x, at (row 0, col 0).") [.../casadi/core/oracle_function.cpp:377]
CasADi - 2024-04-05 22:14:13 WARNING("roots:g failed: Inf detected for output x, at (row 0, col 0).") [.../casadi/core/oracle_function.cpp:377]

[IDAS ERROR]  IDACalcIC
  Newton/Linesearch algorithm failed to converge.


[IDAS ERROR]  IDASolve
  At t = 0 and h = 3.8147e-07, the corrector convergence failed repeatedly or with |h| = hmin.



SolverError: idaklu solver failed

In [ ]:
pybamm.dynamic_plot(sol, output_variables=["concentration of netutral lithium atoms in the SEI [mol.m-3]",
                    "concentration of llithium ions in the SEI [mol.m-3]",
                                           "Potential in the SEI [V]",
                                           "Thickness of SEI [m]"])